# 1D Heat Equation (Inverse Problem) using pinnDE

This notebook demonstrates how to solve an inverse problem for the 1D Heat Equation using `pinnDE`. The goal is to estimate the unknown diffusivity parameter $\alpha$ from synthetic temperature observations.

### Mathematical Formulation
The model is governed by:
$$\alpha \frac{\partial^2 u}{\partial x^2} - \frac{\partial u}{\partial t} = 0, \quad x \in [0, 1], \quad t \in [0, 1]$$

We generate 400 clean observation points inside the domain using the true value of $\alpha = 0.08$.


In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import pinnde as p
import numpy as np
import tensorflow as tf
import shutil


## 1. Parameters & Synthetic Observation Generation

We set the true diffusivity $\alpha = 0.08$ and sample 400 random locations $(x_i, t_i)$ in the domain to simulate temperature observations $u_i$.


In [2]:
alpha = 0.08
u_true = lambda x, t: np.e**(-((np.pi**2) * alpha * t))*np.sin(np.pi*x)
N_data = 400

xdata = np.random.uniform(0, 1, N_data)
tdata = np.random.uniform(0, 1, N_data)
udata = u_true(xdata, tdata)


## 2. Geometry, Boundaries, and Initial Conditions

We define the domain bounds, homogeneous Dirichlet boundary conditions, and sinusoidal initial condition: $$u(x, 0) = \sin(\pi x)$$


In [3]:
timerange = [0, 1]
tre = p.domain.Time_NRect(1, [0], [1], timerange)

bdryfunc = lambda t, x1: 0+t*0
bound = p.boundaries.dirichlet(tre, [bdryfunc])

u0func = lambda x1: tf.sin(np.pi*x1)
inits = p.initials.initials(tre, [u0func])


## 3. Dataset Configuration, Model Definition, and Joint Training

We bundle the domain data and observations using `TimeInvPINNData`. We define $\alpha$ as an unknown trainable parameter and jointly optimize it with the neural network weights over 2000 epochs.


In [4]:
dat = p.data.timeinvpinndata(tre, bound, inits, [tdata, xdata], [udata], 12000, 1000, 1000)

eqn = "alpha*ux1x1 - ut"
mymodel = p.models.invpinn(dat, [eqn], ["alpha"])
mymodel.train(2000)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize_1         │ (None, 1)         │          0 │ input_layer[0][0] │
│ (Normalize)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize_3         │ (None, 1)         │          0 │ input_layer_1[0]… │
│ (Normalize)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 2)         │          0 │ normalize_1[0][0… │
│ (Concatenate)       │                   │            │ normalize_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 60)        │        180 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 60)        │      3,660 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 60)        │      3,660 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 60)        │      3,660 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 1)         │         61 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         61 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 11,282 (44.07 KB)

 Trainable params: 11,282 (44.07 KB)

 Non-trainable params: 0 (0.00 B)

...

CLP loss, BC loss, IV loss, Data loss in 1500th epoch:  0.0001,  0.0000,  0.0000,  0.0002.


CLP loss, BC loss, IV loss, Data loss in 1600th epoch:  0.0001,  0.0000,  0.0000,  0.0001.


CLP loss, BC loss, IV loss, Data loss in 1700th epoch:  0.0001,  0.0000,  0.0000,  0.0001.


CLP loss, BC loss, IV loss, Data loss in 1800th epoch:  0.0001,  0.0000,  0.0000,  0.0001.


CLP loss, BC loss, IV loss, Data loss in 1900th epoch:  0.0001,  0.0000,  0.0000,  0.0001.


## 4. Plotting Solution and Epoch Loss

We output the estimated value of the constant $\alpha$ and use pinnDE plotters to plot the predicted solution profile and loss history. The output plots are saved inside the `figures/` directory.


In [5]:
print(mymodel.get_trained_constants())
p.plotters.plot_solution_prediction_time1D(mymodel)
p.plotters.plot_epoch_loss(mymodel)

# Move generated plots to the figures directory
os.makedirs("figures", exist_ok=True)
if os.path.exists("PDE-solution-pred.png"):
    shutil.move("PDE-solution-pred.png", "figures/PDE-solution-pred.png")
if os.path.exists("PDE-epoch-loss.png"):
    shutil.move("PDE-epoch-loss.png", "figures/PDE-epoch-loss.png")


[array([[0.08365535],
       [0.07227557],
       [0.09044762],
       [0.08130939],
       [0.07917008],
       [0.1156904 ],
       [0.10963403],
       [0.08611698],
       [0.07369889],
       [0.06556236],
       [0.10757603],
       [0.08360985],
       [0.08429851],
       [0.08196142],
       [0.07239401],
       [0.09137158],
       [0.09471034],
       [0.08636367],
       [0.08928914],
       [0.05965363],
       [0.11112724],
       [0.09038065],
       [0.11398739],
       [0.07494073],
       [0.06456272],
       [0.08292376],
       [0.11043759],
       [0.0924006 ],
       [0.10792968],
       [0.08102316],
       [0.09285349],
       [0.08016478],
       [0.05664639],
       [0.07433622],
       [0.09875521],
       [0.09941407],
       [0.07145015],
       [0.07625483],
       [0.10614802],
       [0.07811655],
       [0.07614565],
       [0.07975231],
       [0.09888759],
       [0.08603752],
       [0.10348444],
       [0.08517116],
       [0.09285672],
       [0.05

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

## 5. Estimation Accuracy & Error Analysis

Finally, we extract the estimated value of $\alpha$ and compute the absolute and relative errors compared to the true value.


In [6]:
constants = mymodel.get_trained_constants()
alpha_estime = float(np.mean(constants[0]))

print(f"alpha estime  : {alpha_estime:.5f}")
print(f"alpha reel    : {alpha:.5f}")
print(f"erreur absolue : {abs(alpha_estime - alpha):.5f}")
print(f"erreur relative : {abs(alpha_estime - alpha) / alpha:.2%}")


alpha estime  : 0.08543
alpha reel    : 0.08000
erreur absolue : 0.00543
erreur relative : 6.79%


## Conclusion & Limits of `pinnDE`

- **Usability:** It is very straightforward to initialize, configure, and train simple PINN models using the `pinnDE` library.
- **Lack of Flexibility:** However, `pinnDE` lacks flexibility for defining custom loss weights, advanced boundary condition formulations, or custom neural network architectures even resolving integro-differential equations.
- **Dimensionality Limit:** The library is strictly limited to solving equations in at most **1+2D** dimensions (1 time + 2 space dimensions). This represents a major limitation for our final objective of solving the **2D Radiative Transfer Equation (RTE/RFE)**, which involves an integro-differential equation over 2D space ($x, y$) and 2D direction ($\mu, \eta$), effectively requiring a higher-dimensional formulation. Therefore, we must build our PINN models from scratch using pure PyTorch.
